In [1]:
import pandas as pd

default_df = pd.read_csv(r"C:\Users\hari7\Documents\Anamoly Detection\uploads\uploads\default\1781332375_gasifier 30lpm 0.csv")


In [10]:
import os
import glob
import pandas as pd

def inspect_default_folder(file_path):
    """
    Dynamically locates the table header in Agilent 34970A files, loads the CSV,
    and outputs shape, column structure, and baseline statistics.
    """
    header_row_index = 0
    
    # Locate the telemetry table header row
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for idx, line in enumerate(f):
            if 'Scan Num' in line or '101 (' in line or 'Scan Swee' in line:
                header_row_index = idx
                break
                
    # Load dataset from the detected header row
    df = pd.read_csv(file_path, skiprows=header_row_index)
    
    # Clean column whitespace and drop completely empty rows or columns
    df.columns = df.columns.str.strip()
    df = df.dropna(how='all', axis=1).dropna(how='all', axis=0)
    
    return df

# ==========================================
# EXECUTION ON DEFAULT FOLDER
# ==========================================
default_path = r"C:\Users\hari7\Documents\Anamoly Detection\uploads\uploads\default"
csv_files = glob.glob(os.path.join(default_path, "*.csv"))

print(f"Found {len(csv_files)} files in default folder. Running inspection...\n")

for i, file_path in enumerate(csv_files, 1):
    file_name = os.path.basename(file_path)
    
    try:
        df = inspect_default_folder(file_path)
        
        print(f"File {i}: {file_name}")
        print(f"   Shape: {df.shape[0]} rows by {df.shape[1]} columns")
        print(f"   Columns: {list(df.columns)}")
        print(f"   Missing Values: {df.isnull().sum().sum()} total nulls")
        
        # Select numeric columns for basic baseline stats
        numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
        if len(numeric_cols) > 0:
            print("\n   Baseline Data Summary (First 4 Numeric Columns):")
            print(df[numeric_cols[:4]].describe().loc[['mean', 'min', 'max', 'std']])
        
        print("\n" + "="*65 + "\n")
        
    except Exception as e:
        print(f"Could not read {file_name}: {e}\n")

Found 1 files in default folder. Running inspection...

File 1: 1781332375_gasifier 30lpm 0.csv
   Shape: 456 rows by 7 columns
   Columns: ['Scan Sweep Time (Sec)', 'Scan Number', '101 (°C)', '102 (°C)', '103 (°C)', '104 (°C)', '105 (°C)']
   Missing Values: 0 total nulls

   Baseline Data Summary (First 4 Numeric Columns):
      Scan Number     101 (°C)    102 (°C)    103 (°C)
mean    228.50000   674.904642  649.647536  694.076475
min       1.00000    29.560183   51.896423  595.116976
max     456.00000  1084.429830  902.305949  888.697890
std     131.78012   397.122003  211.825530   54.190740




In [9]:
import os
import glob
import pandas as pd

def load_34970a_sensor_data(file_path):
    """
    Dynamically locates the start of the telemetry table in an Agilent/Keysight 34970A 
    data log file and imports it into a clean Pandas DataFrame.
    """
    header_row_index = None
    
    # 1. Scan the file line-by-line to find where the actual data table begins
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for idx, line in enumerate(f):
            # We look for signature strings present in the telemetry header row
            if 'Scan Num' in line or '101 (' in line or 'Scan Swee' in line:
                header_row_index = idx
                break
                
    if header_row_index is None:
        raise ValueError(f"Could not locate telemetry data header in: {os.path.basename(file_path)}")
        
    # 2. Load the CSV into Pandas, skipping all metadata rows above the detected header
    df = pd.read_csv(file_path, skiprows=header_row_index)
    
    # 3. Clean up column names (strip unexpected spaces or formatting anomalies)
    df.columns = df.columns.str.strip()
    
    # 4. Drop any completely empty rows or columns (often caused by trailing commas in logger exports)
    df = df.dropna(how='all', axis=1).dropna(how='all', axis=0)
    
    # 5. Standardize the Timestamp / Scan Sweep column name for consistency
    time_col_candidates = [col for col in df.columns if 'swee' in col.lower() or 'time' in col.lower()]
    if time_col_candidates:
        df.rename(columns={time_col_candidates[0]: 'Timestamp'}, inplace=True)
        
    return df

# ==========================================
# BATCH EXECUTION ON YOUR 'DEFAULT' FOLDER
# ==========================================
default_path = r"C:\Users\hari7\Documents\Anamoly Detection\uploads\uploads\default"
csv_files = glob.glob(os.path.join(default_path, "*.csv"))

print(f"Successfully located {len(csv_files)} files. Beginning clean data extraction...\n")

for i, file_path in enumerate(csv_files, 1):
    file_name = os.path.basename(file_path)
    try:
        # Use our custom hardware loader
        df_clean = load_34970a_sensor_data(file_path)
        
        print(f"✅ File {i}: {file_name}")
        print(f"   • Clean Shape: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")
        print(f"   • Extracted Headers: {list(df_clean.columns)}")
        print(f"   • First Row Sample Summary:\n{df_clean.iloc[0:2, :4]}\n")
        print("-" * 60)
        
    except Exception as e:
        print(f" Error loading {file_name}: {str(e)}\n")

Successfully located 1 files. Beginning clean data extraction...

✅ File 1: 1781332375_gasifier 30lpm 0.csv
   • Clean Shape: 456 rows × 7 columns
   • Extracted Headers: ['Timestamp', 'Scan Number', '101 (°C)', '102 (°C)', '103 (°C)', '104 (°C)', '105 (°C)']
   • First Row Sample Summary:
                 Timestamp  Scan Number   101 (°C)   102 (°C)
0  2024-05-14 12:18:23.620            1  29.936798  51.896423
1  2024-05-14 12:18:28.620            2  30.002421  52.132352

------------------------------------------------------------


In [1]:
import os
import glob
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =========================================================
# 1. DATA LOADER
# =========================================================
def load_default_file(file_path):
    """
    Dynamically finds the telemetry table header in an Agilent 34970A log
    and loads the 5-channel temperature data into a clean DataFrame.
    """
    header_idx = 0
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for idx, line in enumerate(f):
            if any(key in line for key in ['Scan Num', '101 (', 'Scan Swee']):
                header_idx = idx
                break
                
    df = pd.read_csv(file_path, skiprows=header_idx)
    df.columns = df.columns.str.strip()
    df = df.dropna(how='all', axis=1).dropna(how='all', axis=0)
    
    # Standardize time column name
    for col in df.columns:
        if 'swee' in col.lower() or 'time' in col.lower():
            df.rename(columns={col: 'Timestamp'}, inplace=True)
            break
            
    return df

# =========================================================
# 2. STATISTICAL ENGINE
# =========================================================
def compute_baseline_statistics(df):
    """
    Computes univariate statistics for each sensor feature and global
    multivariate correlation metrics across the entire system.
    """
    # Select only numeric channels (exclude Scan Number and Timestamp)
    sensor_cols = [col for col in df.columns if '°C' in col or 'ch_' in col.lower()]
    sensor_df = df[sensor_cols]
    
    print("================================================================")
    print(" FEATURE-LEVEL STATISTICAL SUMMARY (PER SENSOR)")
    print("================================================================")
    
    stats_df = pd.DataFrame({
        'Mean': sensor_df.mean(),
        'Std Dev': sensor_df.std(),
        'Min': sensor_df.min(),
        '25% (Q1)': sensor_df.quantile(0.25),
        '50% (Median)': sensor_df.median(),
        '75% (Q3)': sensor_df.quantile(0.75),
        'Max': sensor_df.max(),
        'IQR': sensor_df.quantile(0.75) - sensor_df.quantile(0.25),
        'Skewness': sensor_df.skew(),
        'Kurtosis': sensor_df.kurtosis()
    })
    
    print(stats_df.round(4))
    
    print("\n================================================================")
    print(" WHOLE-DATASET STATISTICAL SUMMARY")
    print("================================================================")
    print(f"Total Recorded Scans : {len(df)}")
    print(f"Total Features       : {len(sensor_cols)}")
    print(f"Global Mean Temp     : {sensor_df.values.mean():.4f} °C")
    print(f"Global Temp Spread   : {sensor_df.values.min():.4f} °C to {sensor_df.values.max():.4f} °C")
    print(f"Total Missing Values : {sensor_df.isnull().sum().sum()}")
    
    print("\n--- Inter-Sensor Correlation Matrix (Pearson r) ---")
    correlation_matrix = sensor_df.corr()
    print(correlation_matrix.round(4))
    print("================================================================\n")
    
    return sensor_cols, stats_df, correlation_matrix

# Execute on the first file in your default folder
default_dir = r"C:\Users\hari7\Documents\Anamoly Detection\uploads\uploads\default"
csv_files = glob.glob(os.path.join(default_dir, "*.csv"))

if csv_files:
    sample_file = csv_files[0]
    print(f"Analyzing Baseline File: {os.path.basename(sample_file)}\n")
    df_default = load_default_file(sample_file)
    sensors, feature_stats, corr_matrix = compute_baseline_statistics(df_default)
else:
    print("No CSV files found in the specified path.")

Analyzing Baseline File: 1781332375_gasifier 30lpm 0.csv

 FEATURE-LEVEL STATISTICAL SUMMARY (PER SENSOR)
              Mean   Std Dev       Min  25% (Q1)  50% (Median)  75% (Q3)  \
101 (°C)  674.9046  397.1220   29.5602  111.2509      866.6602  956.5828   
102 (°C)  649.6475  211.8255   51.8964  678.9386      696.2423  719.2182   
103 (°C)  694.0765   54.1907  595.1170  656.3024      685.6056  721.5174   
104 (°C)  592.0016   26.2449  504.2383  576.0455      590.9057  608.1941   
105 (°C)  537.9005   29.4195  473.8383  517.7271      532.9059  553.5970   

                Max       IQR  Skewness  Kurtosis  
101 (°C)  1084.4298  845.3319   -0.8089   -1.1097  
102 (°C)   902.3059   40.2796   -2.0802    3.2374  
103 (°C)   888.6979   65.2150    0.8312    0.6474  
104 (°C)   754.5641   32.1486    0.9554    5.8585  
105 (°C)   735.6866   35.8699    1.5335    5.5963  

 WHOLE-DATASET STATISTICAL SUMMARY
Total Recorded Scans : 456
Total Features       : 5
Global Mean Temp     : 629.7062 °C
Gl

In [2]:
def plot_whole_dataset(df, sensor_cols, corr_matrix):
    """
    Renders the combined multivariate time-series plot and correlation heatmap.
    """
    # 1. Combined Multivariate Time-Series Chart
    fig_time = px.line(
        df, 
        x='Timestamp' if 'Timestamp' in df.columns else df.index, 
        y=sensor_cols,
        title="Whole-Dataset Baseline Telemetry (All Channels)",
        labels={'value': 'Temperature (°C)', 'variable': 'Sensor Channel'}
    )
    fig_time.update_layout(
        hovermode='x unified',
        xaxis=dict(rangeslider=dict(visible=True), type='category')
    )
    fig_time.show()
    
    # 2. Inter-Sensor Correlation Heatmap
    fig_corr = px.imshow(
        corr_matrix, 
        text_auto=True, 
        aspect="auto",
        color_continuous_scale='RdBu_r',
        title="Inter-Sensor Correlation Matrix (Pearson r)",
        labels=dict(color="Correlation")
    )
    fig_corr.show()

def plot_individual_features(df, sensor_cols):
    """
    Renders individual time-series subplots paired with statistical box plots
    for each feature to inspect baseline stability and outlier envelopes.
    """
    num_sensors = len(sensor_cols)
    fig = make_subplots(
        rows=num_sensors, 
        cols=2, 
        column_widths=[0.75, 0.25],
        subplot_titles=[item for sublist in [[f"{col} - Time Series", f"{col} - Distribution"] for col in sensor_cols] for item in sublist],
        horizontal_spacing=0.08,
        vertical_spacing=0.06
    )
    
    colors = px.colors.qualitative.Plotly
    x_axis = df['Timestamp'] if 'Timestamp' in df.columns else df.index
    
    for idx, col in enumerate(sensor_cols):
        row_num = idx + 1
        color = colors[idx % len(colors)]
        
        # Left Column: Individual Time-Series Trend
        fig.add_trace(
            go.Scatter(
                x=x_axis, 
                y=df[col], 
                mode='lines', 
                name=col,
                line=dict(color=color, width=1.5),
                showlegend=False
            ),
            row=row_num, col=1
        )
        
        # Right Column: Statistical Box Plot (Outlier Detection Boundaries)
        fig.add_trace(
            go.Box(
                y=df[col], 
                name=col, 
                marker_color=color,
                boxpoints='outliers',
                showlegend=False
            ),
            row=row_num, col=2
        )
        
        fig.update_yaxes(title_text="Temp (°C)", row=row_num, col=1)
        
    fig.update_layout(
        title_text="Individual Feature Inspection: Telemetry vs. Outlier Box Plots",
        height=240 * num_sensors,
        width=1200,
        showlegend=False
    )
    fig.show()

# Execute interactive plots
if csv_files:
    plot_whole_dataset(df_default, sensors, corr_matrix)
    plot_individual_features(df_default, sensors)